# TP  Conception d'un agent avec Mem0 (Exploration et Experimentation)


## Plan

| Item | Contenu |
|---|---|
| 0 | Installation et configuration |
| 1 | Mem0 : premier contact (exemple guidé) |
| 2 | ADD / UPDATE / DELETE / NOOP |
| 3 | Persistance single-hop |
| 4 | Mémoire multi-hop et temporelle |
| 5 | Mini LLM-as-a-Judge |
| 6 | Mémoire en graphe (Mem0ᵍ) (bonus) |
| 7 | Intégration LangChain |
| 8 | Agent LangGraph avec mémoire |
| 9 | Connexion d'un outil (agent SAV Beqo) |
| 10 | Mesure latence / tokens, comparaison à un baseline |
| 11 | Bilan et pistes pour aller plus loin |


## Item 0 : Installation et configuration


In [5]:
!pip install mem0ai rodiumai langchain langchain-openai langgraph requests tiktoken matplotlib gradio


/usr/lib/python3.12/pty.py:95: DeprecationWarning: This process (pid=76911) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


  Using cached python_multipart-0.0.32-py3-none-any.whl.metadata (2.1 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached markdown_it_py-4.2.0-py3-none-any.whl.metadata (7.4 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 2.6 MB/s  0:00:12 eta 0:00:010:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.8/796.8 kB 3.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 4.3 MB/s  0:00:01m 4.3 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 5.3 MB/s  0:00:02a 0:00:0136m0:00:01:01
Using cached python_multipart-0.0.32-py3-none-any.whl (30 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 6.0 MB/s  0:00:00m 6.7 MB/s eta 0:00:01
Using cached markdown_it_py-4.2.0-py3-none-any.whl (91 kB)
Using cached mdurl-0.1.2-py3-none-any.whl (10.0 kB)
Using cached shellingham-1.5.4-py2.py3-none-any.whl (9.8 kB)
  Attempting uninstall: toml

## Nous allons utiliser RodiumAI, une api unifié de models llms accessible via un paiement mobile money local

In [1]:
import os
from getpass import getpass

#if "RODIUMAI_API_KEY" not in os.environ:
os.environ["OPENAI_API_KEY"] = getpass("Ta clé OpenAI API : ")
os.environ["OPENAI_BASE_URL"] = "https://api.rodiumai.io/v1"

print("Clé chargée :", bool(os.environ.get("OPENAI_API_KEY")))


Ta clé OpenAI API :  ········


Clé chargée : True


## Item 1 : Mem0 : premier contact (exemple guidé)

Voici le **seul** exemple entièrement écrit du notebook. Étudie-le attentivement : c'est le pattern réutilisé (et adapté) pour tous les items suivants.


In [2]:
from mem0 import Memory

# On configure Mem0 : quel LLM extrait/gère les souvenirs, quel modèle transforme le texte en vecteurs.
config = {
    "llm": {"provider": "openai", "config": {"model": "openai/gpt-4o-mini", "temperature": 0.0}},
    "embedder": {"provider": "openai", "config": {"model": "openai/text-embedding-3-small"}},
}

if "memory" not in globals():
    config = {
        "vector_store": {
            "provider": "qdrant",
            "config": {"path": ":memory:"},
        },
        "llm": {"provider": "openai", "config": {"model": "openai/gpt-4o-mini", "temperature": 0.0}},
        "embedder": {"provider": "openai", "config": {"model": "openai/text-embedding-3-small"}},
    }
    memory = Memory.from_config(config)
    print("Nouvelle instance Memory créée.")
else:
    pass

#memory = Memory.from_config(config)

# On ajoute un premier souvenir pour un utilisateur donné.
resultat = memory.add("Mon nom est roger BANK agé de 20 ans", user_id="demo_user")
print(f"Mermory 1 : {resultat}")


autre_resultat = memory.add("Je suis un informaticien béninois mais je travaille à l'étranger", user_id="demo_user")
# On inspecte ce que Mem0 a réellement fait avec ce message.

#Pour ceci aussi
print(f"Mermory 2 : {autre_resultat}")


[PostHog] Multiple active PostHog clients detected for the same project API key and host. Reuse one Posthog instance per app or process when possible to avoid competing background queues and missed shutdown flushes. Multiple clients are supported when intentional.


Nouvelle instance Memory créée.


Failed to load spaCy lemma model: spaCy is not installed. Install it with: pip install mem0ai[nlp]
fastembed not installed - BM25 keyword search disabled. Install it with: pip install "mem0ai[extras]"
Failed to load spaCy full model: spaCy is not installed. Install it with: pip install mem0ai[nlp]


Mermory 1 : {'results': [{'id': 'b0c0e852-edc0-47bd-937e-2e6f46263668', 'memory': "User's name is Roger BANK and he is 20 years old.", 'event': 'ADD'}]}
Mermory 2 : {'results': []}


**Explication (exemple résolu)** — On voit que l'information ajoutée a été interprétée par le modèle GPT défini dans la config pour créer un objet mémoire avec l'événement `ADD`, puisqu'on vient d'ajouter cette information en mémoire. La langue par défaut des LLMs est l'anglais, donc il n'est pas étonnant que les informations stockées soient en anglais.

Les deux opérations `add` sont résumées en une seule mémoire. Si chaque `print` avait été fait après chaque `ADD`, on aurait obtenu des objets mémoire différents.


## Item 2 : Observer ADD / UPDATE / DELETE / NOOP

**Consignes :**
1. Choisis un `user_id` de ton choix.
2. Ajoute un premier fait sur cet utilisateur (n'importe quel sujet : une préférence, une info personnelle...).
3. Ajoute un deuxième message qui **enrichit** ce fait (sans le contredire) — observe si Mem0 fait un `UPDATE`.
4. Ajoute un troisième message qui **contredit clairement** le premier fait — observe le `DELETE`/`UPDATE`.
5. Ajoute un quatrième message qui ne dit **rien de nouveau** (répète une info déjà connue, autrement formulée) — observe le `NOOP`.


In [3]:
# TODO — étape 1 et 2 : premier fait
# user_id = "..."
# resultat_1 = memory.add("...", user_id=user_id)
# print(resultat_1)
resultat_1 = memory.add("Je suis un athlète béninois avec une descendance américaine", user_id="dona")
print(resultat_1)

{'results': [{'id': 'c720e2b3-4cc5-4413-ad73-0336db0a26c6', 'memory': 'User is a Beninese athlete with American descent.', 'event': 'ADD'}]}


In [4]:
# TODO — étape 3 : message qui enrichit le fait précédent
resultat_2 = memory.add("J'ai fait mes entrainements au bénin et gagné beaucoup de championnat béninois.Mes parents qui sont en amérique ne peut que voir mes sessions à la télévision", user_id="dona")
print(resultat_2)

{'results': [{'id': 'a2dc25bf-b81f-4bb9-97d6-b61786fa0d4c', 'memory': 'User trained in Benin and won many Beninese championships. Their parents, who are in America, can only watch their sessions on television.', 'event': 'ADD'}]}


In [5]:
# TODO — étape 4 : message qui contredit le fait initial
resultat_3 = memory.add("Je suis un athlète quebecois originaire de la russie", user_id="dona")
print(resultat_3)

{'results': [{'id': '4774edc2-1643-4cdf-95c2-ffa8210c74a2', 'memory': 'User is a Quebecois athlete originally from Russia.', 'event': 'ADD'}]}


In [6]:
# TODO — étape 5 : message redondant (NOOP attendu)
resultat_4 = memory.add("Je suis un athlète du canada", user_id="dona")
print(resultat_4)

{'results': []}


In [7]:
# TODO — affiche l'état final de la mémoire de cet utilisateur avec memory.get_all(...)
# et vérifie que ce que tu observes correspond à ce que tu attendais.
final_memory = memory.get_all(filters={"user_id": "dona"})
print(f"Etat final de la mémoire {final_memory}")

Etat final de la mémoire {'results': [{'id': '4774edc2-1643-4cdf-95c2-ffa8210c74a2', 'memory': 'User is a Quebecois athlete originally from Russia.', 'hash': 'b1a2370c292f72c3bcf176567f353632', 'metadata': None, 'created_at': '2026-09-05T18:46:01.772974+00:00', 'updated_at': '2026-09-05T18:46:01.772974+00:00', 'user_id': 'dona', 'attributed_to': 'user'}, {'id': 'a2dc25bf-b81f-4bb9-97d6-b61786fa0d4c', 'memory': 'User trained in Benin and won many Beninese championships. Their parents, who are in America, can only watch their sessions on television.', 'hash': 'c9775a4bc6d1c7911f7318f05b64df66', 'metadata': None, 'created_at': '2026-09-05T18:45:55.991463+00:00', 'updated_at': '2026-09-05T18:45:55.991463+00:00', 'user_id': 'dona', 'attributed_to': 'user'}, {'id': 'c720e2b3-4cc5-4413-ad73-0336db0a26c6', 'memory': 'User is a Beninese athlete with American descent.', 'hash': 'b87ede89472e9cfd2a47d907b64439cf', 'metadata': None, 'created_at': '2026-09-05T18:45:46.478683+00:00', 'updated_at': '

In [8]:
hits = memory.search("nationalité de l'utilisateur", filters={"user_id":"dona"}, limit=10)
for h in hits.get("results", hits):
    print(h.get("score"), "-", h.get("memory"))

[PostHog] [FEATURE FLAGS] PostHog feature flags quota limited, resetting feature flag data.  Learn more about billing limits at https://posthog.com/docs/billing/limits-alerts
[PostHog] [FEATURE FLAGS] Quota limit exceeded: [PostHog] Feature flags quota limited (200)


0.44496795024074054 - User is a Quebecois athlete originally from Russia.
0.4308391130778917 - User is a Beninese athlete with American descent.
0.24008179917802713 - User trained in Benin and won many Beninese championships. Their parents, who are in America, can only watch their sessions on television.


**💬 Interprétation**

L'information *« Je suis un athlète du Canada »* n'a pas été ajoutée : une recherche sur la nationalité de l'utilisateur montre que le souvenir *« User is a Quebecois athlete originally from Russia »* obtient un score plus élevé que *« User is a Beninese athlete with American descent »*. Cela confirme que Mem0 a traité ce message comme un `NOOP` : être québécois implique déjà d'être canadien, donc l'information était redondante avec le souvenir précédent. Le fait d'avoir ajouté *« Je suis un athlète québécois originaire de Russie »* après *« Je suis un athlète béninois avec une descendance américaine »* explique aussi pourquoi ce souvenir plus récent obtient un meilleur score de pertinence.


**❓ Question ouverte** — As-tu réussi à provoquer un `NOOP` du premier coup ? Si non, qu'est-ce que ça t'apprend sur la façon dont le LLM sous-jacent juge la "nouveauté" d'une information ?

_Ta réponse ici :_

> 

## Item 3 : Persistance single-hop

Objectif : reproduire, avec ses propres mots, le scénario du papier de recherche (l'exemple "végétarien" — un utilisateur donne une contrainte, puis revient plus tard demander une recommandation qui doit la respecter).

**Consignes :**
1. Écris une fonction `repondre_avec_memoire(user_id, question)` qui :
   - récupère les souvenirs pertinents via `memory.search(...)`,
   - construit un prompt qui inclut ce contexte,
   - appelle le LLM (`OpenAI` ou `ChatOpenAI`) pour générer une réponse,
   - retourne cette réponse.
2. Écris aussi une fonction `repondre_sans_memoire(question)` qui n'utilise **aucun** contexte.
3. Teste les deux fonctions sur la même question de suivi, après avoir enregistré une contrainte au préalable.


In [9]:
from openai import OpenAI

client = OpenAI()


def repondre_avec_memoire(user_id: str, question: str) -> str:
    """TODO : documente ici ce que fait ta fonction, étape par étape."""
    # TODO
    preferences_hits = memory.search("Préférences de l'utilsateur en matière d'alimentation", filters={"user_id": user_id})
    preferences = preferences_hits.get("results", preferences_hits)[0]
    preference_pertinent = preferences.get("memory")
    response = client.responses.create(
        model="openai/gpt-4o-mini",
        instructions=f"Tu es un assistant personnel qui aide à prendre des décisions concernant une bonne alimentation pour une bonne santé. {preference_pertinent}",
        input=question
    )
    return response.output_text


def repondre_sans_memoire(question: str) -> str:
    """TODO"""
    # TODO
    response = client.responses.create(
        model="openai/gpt-4o-mini",
        instructions=f"Tu es un assistant personnel qui aide à prendre des décisions concernant une bonne alimentation pour une bonne santé",
        input=question
    )
    return response.output_text


In [10]:
# TODO — enregistre une contrainte (régime, préférence, contrainte de planning, ce que tu veux)
memory.add("Je fais du régime et j'évite au maximum les aliments gras", user_id="johndoe")

{'results': [{'id': '7a4d5cd8-2f6e-47f8-bea1-117d695a40ac',
   'memory': 'User is on a diet and is avoiding fatty foods as much as possible.',
   'event': 'ADD'},
  {'id': '3cd5c62b-9f28-45c8-bd9b-1bc4b350366d',
   'memory': 'User has been confirmed by a doctor to have lost some fat and is in good health.',
   'event': 'ADD'},
  {'id': '9473bf31-db86-4c9a-8b46-1e755a16e08f',
   'memory': 'User is becoming increasingly fit, with visible abs.',
   'event': 'ADD'},
  {'id': '49595e9f-2a13-4253-9b03-d05a2b7a6ea2',
   'memory': 'User has noticed that girls at their college are attracted to them.',
   'event': 'ADD'}]}

In [11]:
# TODO — pose la même question de suivi aux deux fonctions et affiche les deux réponses côte à côte
print("############## REPONDRE AVEC MEMOIRE ##################")
print(repondre_avec_memoire("johndoe", "Donne moi des idées d'aliments pour mon diner"))

print("############## REPONDRE SANS MEMOIRE ##################")
print(repondre_sans_memoire("Donne moi des idées d'aliments pour mon diner"))

############## REPONDRE AVEC MEMOIRE ##################
Bien sûr ! Voici quelques idées de dîners sains et pauvres en matières grasses :

1. **Filet de poulet grillé avec légumes vapeur** : Assaisonne-le avec des herbes aromatiques pour plus de saveur.

2. **Poisson blanc au four (comme le cabillaud ou le tilapia)** : Accompagne-le de brocoli et de carottes rôties.

3. **Salade de quinoa** : Ajoute des tomates cerises, du concombre, des poivrons et des herbes fraîches avec un jus de citron en vinaigrette.

4. **Soupe de lentilles** : Prépare-la avec des légumes comme des carottes, du céleri et des épices.

5. **Wraps de laitue** : Remplis-les de dinde hachée ou de tofu sauté avec des légumes croquants.

6. **Omelette aux épinards et champignons** : Ajoute des blancs d'œufs pour réduire les graisses.

7. **Bowl de légumes rôtis** : Utilise des patates douces, des courgettes, et des pois chiches, et arrose d'un peu de sauce au yaourt.

8. **Sauté de poulet aux légumes** : Utilise beaucou

**💬 Interprétation**

La réponse avec mémoire fait clairement ressortir que l'utilisateur suit un régime et évite les aliments gras, alors que la réponse sans mémoire cite des aliments sans tenir compte de ce contexte. Cela s'explique par le fait que `memory.add(...)` a enregistré cette contrainte comme un souvenir distinct, que `memory.search(...)` a ensuite retrouvé et injecté dans le prompt système sous une forme exploitable par le LLM — ce qui lui a permis de générer une réponse respectant réellement le contexte de l'utilisateur.


## Item 4 : Mémoire multi-hop et raisonnement temporel

**Consignes :**
1. Simule au moins 3 "sessions" espacées dans le temps pour un même utilisateur (utilise `time.sleep(...)` entre chaque `memory.add`), chacune apportant un fait distinct mais reliable aux autres (ex : un contexte professionnel qui évolue, un projet qui avance par étapes...).
2. Pose une question **multi-hop** : une question dont la réponse nécessite de combiner au moins deux faits enregistrés séparément.
3. Pose une question **temporelle** : une question qui demande de savoir *quand* ou *dans quel ordre* les choses se sont passées.
4. Pour chaque question, indique **avant** de l'exécuter ce que l'on attend comme réponse correcte, puis compare avec ce que le système a réellement produit.


In [12]:
# TODO — construis tes 3+ sessions espacées dans le temps
import time

memory.add("Je suis passé après chez le médécin et il a confirmé que j'ai perdu un peu de graisse et mon corps se porte bien", user_id="johndoe")
time.sleep(3)
memory.add("Je deviens de plus en plus fit et mes abdos deviennent visibles", user_id="johndoe")
time.sleep(4)
memory.add("les filles de mon collège sont atirées par moi", user_id="johndoe")

# On va changer la fonction en passant cette fois ci la question dans le search pour resortir une information plus globale qui va servir dans le prompt

def repondre_avec_memoire(user_id: str, question: str) -> str:
    """TODO : documente ici ce que fait ta fonction, étape par étape."""
    # TODO
    preferences_hits = memory.search(question, filters={"user_id": user_id})
    preferences = preferences_hits.get("results", preferences_hits)
    top_3 = preferences[:3]
    faits_pertinents = [r.get("memory") for r in top_3]
    texte_faits = ", ".join(faits_pertinents)
    print("##############################")
    print(texte_faits)
    
    response = client.responses.create(
        model="openai/gpt-4o-mini",
        instructions=f"Tu es un assistant personnel. {texte_faits}",
        input=question
    )
    return response.output_text

In [13]:
# TODO — question multi-hop : formule-la, note ta réponse attendue en commentaire, puis interroge le système
reponse = repondre_avec_memoire("johndoe", "Pourquoi les filles deviennent de plus en plus attirée par moi")
print(reponse)

##############################
User has noticed that girls at their college are attracted to them., User is becoming increasingly fit, with visible abs.
Il y a plusieurs raisons pour lesquelles les filles peuvent être de plus en plus attirées par toi :

1. **Confiance en soi** : Si tu te sens bien dans ta peau et que tu es confiant, cela peut se ressentir et attirer les autres vers toi.

2. **Changements physiques** : Comme tu l'as mentionné, devenir de plus en plus fit et avoir des abdos visibles peut augmenter ton attractivité physique.

3. **Attitude positive** : Une attitude amicale et ouverte peut également rendre une personne plus attirante. Les gens aiment se sentir bien en présence de quelqu'un.

4. **Comportement social** : Si tu interagis davantage avec les autres et que tu crées des liens, cela peut augmenter l'intérêt des filles.

5. **Évolution personnelle** : Travailler sur soi, que ce soit sur le plan physique ou émotionnel, peut rendre quelqu'un plus attrayant.

Prends 

In [14]:
# TODO — question temporelle : même démarche
reponse = repondre_avec_memoire("johndoe","qu'est-ce qui s'est passé en premier pour que je devienne plus séduisant ?")
print(reponse)

##############################
User has noticed that girls at their college are attracted to them., User is becoming increasingly fit, with visible abs., User has been confirmed by a doctor to have lost some fat and is in good health.
Il y a plusieurs facteurs qui ont pu contribuer à ton augmentation d'attractivité :

1. **Santé et condition physique** : Ta perte de poids et ton amélioration de la forme physique ont sans doute joué un rôle majeur. Être en bonne santé et avoir une silhouette musclée peut augmenter la confiance en soi, ce qui est souvent perçu comme séduisant.

2. **Confiance en soi** : Lorsque tu te sens bien dans ta peau, cela se reflète dans ta manière de te comporter. Une attitude confiante attire souvent les autres.

3. **Style personnel** : Si tu as aussi modifié ton style vestimentaire ou ta présentation personnelle, cela pourrait avoir un impact sur la façon dont les autres te perçoivent.

4. **Interaction sociale** : Un comportement amical et engageant peut atti

**💬 Interprétation**

La question multi-hop fait ressortir la fitness et les abdos comme explications de l'attirance des filles, mais pas la visite chez le médecin — alors que la question temporelle, elle, mobilise l'ensemble des informations, y compris la visite médicale. Cela s'explique par le fonctionnement de `memory.search(...)` : la recherche sémantique n'a retourné que les 2 souvenirs les plus proches de la question posée, la visite chez le médecin n'ayant pas de lien sémantique direct avec l'attirance. La question temporelle, en demandant explicitement une chronologie, a en revanche entraîné la récupération de l'ensemble des faits liés — au point de faire apparaître une date qui n'avait pourtant pas été mentionnée explicitement au préalable.


## Item 5 : Mini LLM-as-a-Judge

**Consignes :**
1. Écris une fonction `juger_reponse(question, reponse_attendue, reponse_generee)` qui utilise un LLM pour juger si la réponse générée est correcte. **Prompt d'évaluation rédigé soi-même** — en réfléchissant à ce qui doit être précisé pour que le jugement soit fiable (tolérance de formulation, gestion des dates relatives, etc., comme discuté dans le résumé du papier).
2. Construis un mini jeu de test d'au moins 6 questions couvrant les 4 catégories vues dans le papier (single-hop, multi-hop, temporal, open-domain).
3. Calcule un taux de réussite global et un taux par catégorie.


In [24]:
import json
def juger_reponse(question: str, reponse_attendue: str, reponse_generee: str) -> dict:
    """TODO : écris ton propre prompt de jugement. Réfléchis à :
    - comment gérer une réponse plus longue mais correcte,
    - comment gérer les formulations de dates différentes,
    - le format de sortie que tu veux (JSON recommandé pour pouvoir le parser).
    """
    # TODO
    judge_prompt = f"""Tu es chargé de scoré des réponses générées par CORRECT ou INCORRECT.
    Tu auras en entrée la question ou la demande de l'utilisateur, la bonne réponse correspondante et la réponse généré.
    Par exemple :
    Question : Quand est ce que j'ai souscrit à l'abonnement de claude pour la première fois
    Bonne réponse : Vous avez souscrit à l'abonnement claude le 01 Septembre 2026
    La réponse générée peut ne pas etre exact mot à mot ou plus long mais tu ne seras pas trop strict sur le jugement
    Concernant les dates, ils doivent etre au format alphanumérique. C'est à dire que si dans une réponse généré il doit avoir une date cela doit être
    au format alphanumérique . Une date comme 01 Sept 2026 est acceptable
    
    En sortie tu envoie un json qui comporte le jugement (CORRECT, INCORRECT) et la justification comme ceci
    {{"label": "CORRECT/INCORRECT", "description": "la justification"}}. La justification doit préciser clairement pourquoi la réponse a été libellé
    de cette manière. Pas de commentaire autour. Tu génère le json finale
    Voici une entrée :
    Question : {question}
    Bonne réponse: {reponse_attendue}
    Reponse  généré : {reponse_generee}
    """

    response = client.responses.create(
        model="openai/gpt-4o-mini",
        input=judge_prompt,
        text={
        "format": {
            "type": "json_schema",
            "name": "verdict_juge",
            "strict": True,
            "schema": {
                "type": "object",
                "properties": {
                    "label": {"type": "string", "enum": ["CORRECT", "INCORRECT"]},
                    "description": {"type": "string"}
                },
                "required": ["label", "description"],
                "additionalProperties": False

            }
        }
    }
    )
    try:
        response_json = json.loads(response.output_text)
    except (json.JSONDecodeError, TypeError, AttributeError):
        response_json = {}

    return response_json


In [25]:
# TODO — construis ton jeu de test structuré, par exemple une liste de dicts avec
# {"categorie": "single-hop", "user_id": ..., "question": ..., "reponse_attendue": ...}

# On oriente notre jeu de données
jeu_de_donnees = [
    {
        "categorie": "single-hop",
        "user_id": "holding_corp",
        "question": "Quel nombre de participants avons-nous interviewé aujourd'hui ?",
        "reponse_attendue": "Vous avez interviewé 20 participants aujourd'hui"
    },
    {
        "categorie": "single-hop",
        "user_id": "holding_corp",
        "question": "Quelle est la description de l'offre que nous proposons ?",
        "reponse_attendue": "L'offre concerne un poste d'assistant comptable pour les besoins immédiats de votre entreprise Holding Corp"
    },
    {
        "categorie": "single-hop",
        "user_id": "holding_corp",
        "question": "Quel salaire mensuel proposons-nous pour ce poste ?",
        "reponse_attendue": "Vous proposez un salaire mensuel de 150 000 FCFA pour le poste d'assistant comptable"
    },
    {
        "categorie": "multi-hop",
        "user_id": "holding_corp",
        "question": "Le nombre de candidats reçus aujourd'hui semble-t-il suffisant pour le poste que nous cherchons à pourvoir en urgence ?",
        "reponse_attendue": "20 candidats ont été interviewés aujourd'hui pour le poste d'assistant comptable, un besoin immédiat de Holding Corp — ce volume paraît suffisant pour couvrir un poste unique en urgence."
    },
    {
        "categorie": "multi-hop",
        "user_id": "holding_corp",
        "question": "Avec 20 candidats déjà reçus aujourd'hui, reste-t-il de la marge avant la clôture des candidatures le 15 septembre ?",
        "reponse_attendue": "Oui, avec 20 candidats déjà interviewés aujourd'hui et une clôture fixée au 15 septembre 2026, il reste encore du temps pour recevoir d'autres candidatures avant la fin du processus."
    },
    {
        "categorie": "multi-hop",
        "user_id": "holding_corp",
        "question": "Le salaire mensuel proposé est-il cohérent avec un poste à pourvoir en urgence ?",
        "reponse_attendue": "Le salaire de 150 000 FCFA/mois est proposé pour le poste d'assistant comptable, correspondant à un besoin de recrutement immédiat chez Holding Corp."
    },
    {
        "categorie": "temporal",
        "user_id": "holding_corp",
        "question": "Le nombre de participants interviewés a-t-il augmenté ou diminué par rapport à la semaine dernière ?",
        "reponse_attendue": "Le nombre de participants interviewés a augmenté : vous étiez passés de 15 participants la semaine dernière à 20 aujourd'hui."
    },
    {
        "categorie": "temporal",
        "user_id": "holding_corp",
        "question": "Quand aura lieu le prochain tour d'entretiens, et se situe-t-il avant ou après la clôture des candidatures ?",
        "reponse_attendue": "Le prochain tour d'entretiens est prévu le 20 septembre 2026, soit après la clôture des candidatures fixée au 15 septembre 2026."
    },
    {
        "categorie": "temporal",
        "user_id": "holding_corp",
        "question": "Depuis combien de temps le poste d'assistant comptable est-il ouvert au recrutement, jusqu'à la clôture des candidatures ?",
        "reponse_attendue": "Le recrutement a débuté il y a une semaine (15 participants interviewés) et se poursuit jusqu'au 15 septembre 2026, date de clôture des candidatures."
    },
    {
        "categorie": "open-domain",
        "user_id": "holding_corp",
        "question": "Où en est notre processus de recrutement en ce moment ?",
        "reponse_attendue": "Vous recrutez actuellement un assistant comptable pour un besoin immédiat chez Holding Corp ; 20 participants ont été interviewés aujourd'hui, contre 15 la semaine dernière, ce qui suggère un processus en accélération."
    },
    {
        "categorie": "open-domain",
        "user_id": "holding_corp",
        "question": "Peux-tu me faire un état complet du recrutement en cours ?",
        "reponse_attendue": "Le recrutement concerne un poste d'assistant comptable pour un besoin immédiat chez Holding Corp, avec un salaire mensuel de 150 000 FCFA. 20 candidats ont été interviewés aujourd'hui (contre 15 la semaine dernière), les candidatures se clôturent le 15 septembre 2026, et le tour final d'entretiens est prévu le 20 septembre 2026."
    },
    {
        "categorie": "open-domain",
        "user_id": "holding_corp",
        "question": "Que dois-je savoir avant mon prochain point sur ce recrutement ?",
        "reponse_attendue": "Le poste d'assistant comptable chez Holding Corp reste ouvert jusqu'au 15 septembre 2026 ; 20 candidats ont déjà été interviewés (contre 15 la semaine précédente) pour un salaire mensuel de 150 000 FCFA, et le prochain tour d'entretiens aura lieu le 20 septembre 2026."
    }
]

In [27]:
# TODO - boucle d'évaluation : génère une réponse, juge-la, stocke le verdict, puis calcule les taux par catégorie
system_prompt = """Tu es l'assistant virtuel de recrutement de Holding Corp.
Tu réponds aux questions du RH en te basant uniquement sur les informations disponibles dans ta mémoire (aucune invention, aucune supposition).
Si une information demandée n'est pas dans ta mémoire, dis-le clairement plutôt que de deviner.
Réponds de manière concise et factuelle, en français, en formulant des phrases complètes (pas de liste à puces sauf si explicitement demandé)."""

import time
from collections import defaultdict


user_id = "holding_corp"

# --- Il y a une semaine ---
memory.add("La semaine dernière, Holding Corp a interviewé 15 participants pour le poste d'assistant comptable.", user_id=user_id)
time.sleep(2)

memory.add("Holding Corp propose un poste d'assistant comptable pour un besoin immédiat de l'entreprise.", user_id=user_id)
time.sleep(2)

memory.add("Le salaire mensuel proposé pour le poste d'assistant comptable est de 150 000 FCFA.", user_id=user_id)
time.sleep(2)

memory.add("Les candidatures pour le poste d'assistant comptable se clôturent le 15 septembre 2026.", user_id=user_id)
time.sleep(2)

memory.add("Le prochain tour d'entretiens pour le poste d'assistant comptable est prévu le 20 septembre 2026.", user_id=user_id)
time.sleep(2)

# --- Aujourd'hui ---
memory.add("Aujourd'hui, Holding Corp a interviewé 20 participants pour le poste d'assistant comptable.", user_id=user_id)

def generer_reponse(question, user_id="holding_corp"):
    # integrer la question dans le search
    results_hits = memory.search(question, filters={"user_id": user_id})
    
    #extraire les 4 idées les plus pertinentes par rapport à notre question

    results = results_hits.get("results", results_hits)
    forth_results = results[:4]
    forth_results_memory = [r.get("memory") for r in forth_results]
    forth_results_memory_text = ", ".join(forth_results_memory)

    #former le system prompt à l'aide des prompts de l'aide 
    instructions = system_prompt + forth_results_memory_text
    response = client.responses.create(
        model="openai/gpt-4o-mini",
        instructions=instructions,
        input=question
    )

    return response.output_text


responses_jeu_de_donnes = []

def evaluate(sample_data):
    totaux_par_categorie = defaultdict(int)
    corrects_par_categorie = defaultdict(int)

    for donnee in sample_data:
        question = donnee.get("question")
        reponse_attendue = donnee.get("reponse_attendue")
        reponse = generer_reponse(donnee.get("question"))

        jugement = juger_reponse(question, reponse_attendue, reponse)
        donnee["reponse_genere"] = reponse
        donnee["label"] = jugement.get("label")
        donnee["justification"] = jugement.get("description")
        responses_jeu_de_donnes.append(donnee)

        categorie = donnee.get("categorie")
        totaux_par_categorie[categorie] += 1
        if jugement.get("label") == "CORRECT":
            corrects_par_categorie[categorie] += 1

    data = [
        {"categorie": cat, "score": corrects_par_categorie[cat] / total}
        for cat, total in totaux_par_categorie.items()
    ]

    return data

evaluations_by_categories = evaluate(jeu_de_donnees)

# Affichage détaillé de chaque question évaluée
for r in responses_jeu_de_donnes:
    print(f"[{r['categorie']}] {r['label']}")
    print(f"  Question        : {r['question']}")
    print(f"  Réponse attendue : {r['reponse_attendue']}")
    print(f"  Réponse générée  : {r['reponse_genere']}")
    print(f"  Justification    : {r['justification']}")
    print("-" * 60)

print(evaluations_by_categories)

[single-hop] CORRECT
  Question        : Quel nombre de participants avons-nous interviewé aujourd'hui ?
  Réponse attendue : Vous avez interviewé 20 participants aujourd'hui
  Réponse générée  : Aujourd'hui, nous avons interviewé 20 candidats pour le poste d'assistant comptable.
  Justification    : La réponse générée mentionne correctement le nombre de participants interviewés (20), bien que le terme utilisé soit 'candidats' au lieu de 'participants', ce qui n'affecte pas la précision de l'information.
------------------------------------------------------------
[single-hop] CORRECT
  Question        : Quelle est la description de l'offre que nous proposons ?
  Réponse attendue : L'offre concerne un poste d'assistant comptable pour les besoins immédiats de votre entreprise Holding Corp
  Réponse générée  : L'offre proposée par Holding Corp est pour un poste d'assistant comptable, avec un besoin immédiat au sein de l'entreprise. Le salaire mensuel pour ce poste est de 150,000 FCFA.
  

**💬 Interprétation**

Bien qu'on ne puisse pas tirer de conclusion rigoureuse compte tenu du faible nombre de données évaluées, les catégories single-hop et open-domain obtiennent un score de 100 % tandis que les autres atteignent 66 %.

Pour le multi-hop, le juge estime que la réponse à *« Le nombre de candidats reçus aujourd'hui semble-t-il suffisant... »* n'affirme pas assez clairement que 20 candidats est suffisant — pourtant l'assistant disposait des bons souvenirs, ce qui indique un problème de formulation plutôt que de recherche mémoire.

Pour le temporal, la réponse attendue supposait que « le recrutement a débuté il y a une semaine », une information jamais stockée telle quelle en mémoire (ADD) — l'assistant a d'ailleurs fait preuve de plus de rigueur en indiquant ne pas disposer de cette information.

Ce n'est donc pas que Mem0 gère mal le multi-hop ou vectorise mal l'information, mais plutôt que le juge se montre peu indulgent sur la formulation, et que la vérité attendue suppose parfois une inférence que la mémoire ne contient pas réellement.


## Item 6 : Mémoire en graphe (Mem0ᵍ) (bonus)

Nécessite une instance Neo4j (locale via Docker, ou Neo4j Aura Free) :

```bash
docker run -d -p 7474:7474 -p 7687:7687 -e NEO4J_AUTH=neo4j/password neo4j:latest
```

**Consignes :**
1. Configure une seconde instance `Memory` avec un `graph_store` Neo4j.
2. Réutilise le jeu de faits impliquant plusieurs entités reliées (personnes, lieux, événements) de la partie 4.
3. Compare la réponse obtenue à une question relationnelle avec la mémoire classique vs la mémoire en graphe.


In [ ]:
# TODO — configuration Mem0 avec graph_store Neo4j (adapte url/username/password)


In [ ]:
# TODO — même question posée aux deux mémoires (classique vs graphe), affichage côte à côte


**🔍 Constat** — *Qu'observes-tu exactement ? Décris factuellement ce que tu vois (sans interpréter encore) dans la cellule ci-dessous.*

_Ta réponse ici :_

> 

**💬 Interprétation** — *Pourquoi ce comportement se produit-il, à ton avis ? Relie-le au fonctionnement interne de Mem0 vu dans le résumé de l'article.*

_Ta réponse ici :_

> 

## Item 7 : Intégration LangChain

**Consignes :**
1. Crée une classe qui hérite de `langchain_core.chat_history.BaseChatMessageHistory`, dont les méthodes `add_message` et `messages` s'appuient respectivement sur `memory.add(...)` et `memory.get_all(...)`.
2. Branche cette classe dans un `RunnableWithMessageHistory` autour d'un simple prompt + LLM.
3. Teste une conversation sur plusieurs tours et vérifie que le contexte est bien conservé entre les appels, **sans** avoir à repasser l'historique manuellement.

Documentation : https://python.langchain.com/docs/how_to/message_history/


In [3]:
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

class Mem0ChatMessageHistory(BaseChatMessageHistory):
    def __init__(self, user_id: str):
        self.user_id = user_id

    @property
    def messages(self) -> list:
        results = memory.get_all(filters={"user_id":self.user_id})
        results = results.get("results", results)
        return [HumanMessage(content=r.get("memory", "")) for r in results]

    def add_message(self, message: BaseMessage) -> None:
        memory.add(message.content, user_id=self.user_id)

    def clear(self) -> None:
        memory.delete_all(user_id=self.user_id)

store = {}
def get_session_history(session_id: str) -> Mem0ChatMessageHistory:
    if session_id not in store:
        store[session_id] = Mem0ChatMessageHistory(user_id=session_id)
    return store[session_id]



In [6]:
# TODO — assemble ta chaîne LangChain (prompt + llm) avec RunnableWithMessageHistory
# puis fais 2-3 tours de conversation successifs pour vérifier la persistance du contexte
import gradio as gr

llm = ChatOpenAI(model="openai/gpt-4o-mini")
prompt = ChatPromptTemplate.from_messages([
    ("system", "Tu es un assistant utile."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])
chain = prompt | llm | StrOutputParser()
chain_with_history = RunnableWithMessageHistory(
    chain, get_session_history,
    input_messages_key="input", history_messages_key="history",
)
config = {"configurable": {"session_id": "test_langchain"}}

def repondre_gradio(message, historique):
    return chain_with_history.invoke({"input": message}, config=config)

demo = gr.ChatInterface(fn=repondre_gradio, title="Test LangChain + Mem0")
demo.launch(inline=True)

/home/davakan/jupyter-env/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


/usr/lib/python3.12/contextlib.py:137: Pandas4Warning: 'future.no_silent_downcasting' is deprecated, please refrain from using it.
  return next(self.gen)
/home/davakan/jupyter-env/lib/python3.12/site-packages/gradio/queueing.py:178: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers copies until necessary. Use .copy() to make an eager copy if necessary.
  pd.DataFrame(records).fillna(value=np.nan).infer_objects(copy=False)  # type: ignore
/usr/lib/python3.12/contextlib.py:144: Pandas4Warning: 'future.no_silent_downcasting' is deprecated, please refrain from using it.
  next(self.gen)
/usr/lib/python3.12/contextlib.py:137: Pandas4Warning: 'future.no_silent_downcasting' is deprecated, please refrain from using it.
  return next(self.gen)
/home/davakan/jupyter-env/lib/python3.12/site-packages/gradio/queueing.py:178: Pandas4Warning: The copy keyword is deprecat

**🔍 Constat** — *Qu'observes-tu exactement ? Décris factuellement ce que tu vois (sans interpréter encore) dans la cellule ci-dessous.*

_Ta réponse ici :_

> 

**💬 Interprétation** — *Pourquoi ce comportement se produit-il, à ton avis ? Relie-le au fonctionnement interne de Mem0 vu dans le résumé de l'article.*

_Qu'est-ce que cette intégration t'apporte par rapport à l'appel manuel de `memory.search`/`memory.add` que tu faisais en partie 3 ? Quels sont les inconvénients éventuels ?_

_Ta réponse ici :_

> 

## Item 8 : Agent LangGraph avec mémoire persistante

**Consignes :**
1. Définis un état (`TypedDict`) contenant au minimum : les messages de la conversation, l'identifiant utilisateur, et le contexte mémoire récupéré.
2. Construis un graphe (`StateGraph`) avec au moins 3 nœuds : récupération de mémoire → génération de réponse → écriture en mémoire.
3. Compile et teste l'agent sur au moins deux tours de conversation.
4. Décris (en commentaire) le graphe construit, comme si on devait l'expliquer à quelqu'un qui ne connaît pas LangGraph.


In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
import gradio as gr

class AgentState(TypedDict):
    user_id: str
    question: str
    memoires: list
    reponse: str

def noeud_recuperer_memoire(state: AgentState) -> dict:
    results = memory.search(state["question"], filters={"user_id": state["user_id"]})
    results = results.get("results", results)
    return {"memoires": [r.get("memory") for r in results[:4]]}

def noeud_generer_reponse(state: AgentState) -> dict:
    contexte = ", ".join(state["memoires"])
    instructions = system_prompt + contexte
    response = client.responses.create(model="gpt-4o-mini", instructions=instructions, input=state["question"])
    return {"reponse": response.output_text}

def noeud_ecrire_memoire(state: AgentState) -> dict:
    memory.add(state["question"], user_id=state["user_id"])
    memory.add(state["reponse"], user_id=state["user_id"])
    return {}

graph = StateGraph(AgentState)
graph.add_node("recuperer_memoire", noeud_recuperer_memoire)
graph.add_node("generer_reponse", noeud_generer_reponse)
graph.add_node("ecrire_memoire", noeud_ecrire_memoire)
graph.set_entry_point("recuperer_memoire")
graph.add_edge("recuperer_memoire", "generer_reponse")
graph.add_edge("generer_reponse", "ecrire_memoire")
graph.add_edge("ecrire_memoire", END)
agent_graph = graph.compile()

In [ ]:
# TODO — teste l'agent sur au moins deux tours successifs (deux .invoke() qui partagent le même user_id)
def repondre_gradio_graph(message, historique):
    resultat = agent_graph.invoke({"user_id": "holding_corp", "question": message, "memoires": [], "reponse": ""})
    return resultat["reponse"]

demo = gr.ChatInterface(fn=repondre_gradio_graph, title="Agent LangGraph + Mem0")
demo.launch(inline=True)

**🔍 Constat** — *Qu'observes-tu exactement ? Décris factuellement ce que tu vois (sans interpréter encore) dans la cellule ci-dessous.*

_Ta réponse ici :_

> 

**💬 Interprétation** — *Pourquoi ce comportement se produit-il, à ton avis ? Relie-le au fonctionnement interne de Mem0 vu dans le résumé de l'article.*

_Ta réponse ici :_

> 

## Item 9 : Connexion d'un outil (agent SAV Beqo)

Le périmètre de cet agent n'est pas à choisir librement : il s'agit de construire un agent de service après-vente pour l'agence de voyage **Beqo**, dont la mission unique est la **gestion des réclamations et des plaintes clients**. Rien d'autre.

**Consignes :**
1. Définis un mini schéma de données pour une réclamation (ex. : identifiant client, objet du voyage concerné, nature de la plainte, statut).
2. Écris la fonction qui enregistre une réclamation, puis déclare-la comme `@tool` LangChain avec une docstring précise sur le périmètre strict de l'agent (réclamations uniquement) et sur le moment où l'outil doit être déclenché.
3. Branche cet outil dans l'agent LangGraph via `bind_tools` + `ToolNode` + une route conditionnelle (`tools_condition`), en veillant à ce que le graphe reboucle vers le nœud de génération après l'appel.
4. Teste trois cas : un message qui est clairement une réclamation (ex. bagage perdu, vol annulé, remboursement non reçu) et **doit** déclencher l'outil ; un message hors périmètre (ex. nouvelle réservation) qui **ne doit pas** le déclencher ; un message ambigu, à la frontière entre les deux (ex. question sur les conditions d'annulation) — c'est ce troisième cas qui révèle si le périmètre de l'agent est bien défini.


In [ ]:
from langchain_core.tools import tool
from pydantic import BaseModel
from langgraph.prebuilt import ToolNode, tools_condition
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
import gradio as gr

class Reclamation(BaseModel):
    identifiant_client: str
    objet_voyage: str
    nature_plainte: str
    statut: str = "ouverte"

@tool
def enregistrer_reclamation(identifiant_client: str, objet_voyage: str, nature_plainte: str) -> str:
    """Enregistre une réclamation client pour l'agence de voyage Beqo.
    À utiliser UNIQUEMENT si le message concerne une plainte sur un voyage déjà réservé
    (bagage perdu, vol annulé, remboursement non reçu). Ne pas utiliser pour une nouvelle
    réservation ou une question générale (ex. conditions d'annulation)."""
    reclamation = Reclamation(identifiant_client=identifiant_client, objet_voyage=objet_voyage, nature_plainte=nature_plainte)
    return f"Réclamation enregistrée : {reclamation.model_dump()}"

class BeqoState(TypedDict):
    messages: Annotated[list, add_messages]

In [ ]:
# TODO — reconstruis ton graphe LangGraph avec bind_tools + ToolNode + tools_condition
llm_with_tools = llm.bind_tools([enregistrer_reclamation])

def noeud_agent_beqo(state: BeqoState) -> dict:
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

graph_beqo = StateGraph(BeqoState)
graph_beqo.add_node("agent", noeud_agent_beqo)
graph_beqo.add_node("tools", ToolNode([enregistrer_reclamation]))
graph_beqo.set_entry_point("agent")
graph_beqo.add_conditional_edges("agent", tools_condition)
graph_beqo.add_edge("tools", "agent")
agent_beqo = graph_beqo.compile()

In [ ]:
# TODO — 3 tests : (1) doit déclencher l'outil, (2) ne doit pas, (3) mémoire + outil combinés
conversation_beqo = {"messages": []}

def repondre_gradio_beqo(message, historique):
    conversation_beqo["messages"].append(("user", message))
    resultat = agent_beqo.invoke(conversation_beqo)
    conversation_beqo["messages"] = resultat["messages"]

    derniere_reponse = conversation_beqo["messages"][-1]
    contenu = derniere_reponse.content if hasattr(derniere_reponse, "content") else derniere_reponse[1]
    return contenu

demo_beqo = gr.ChatInterface(fn=repondre_gradio_beqo, title="Agent SAV Beqo")
demo_beqo.launch(inline=True)

**🔍 Constat** — *Qu'observes-tu exactement ? Décris factuellement ce que tu vois (sans interpréter encore) dans la cellule ci-dessous.*

_Dans le test (2), l'agent a-t-il bien évité d'appeler l'outil inutilement ? Si non, qu'aurais-tu pu changer dans la docstring de l'outil pour clarifier son usage ?_

_Ta réponse ici :_

> 

**💬 Interprétation** — *Pourquoi ce comportement se produit-il, à ton avis ? Relie-le au fonctionnement interne de Mem0 vu dans le résumé de l'article.*

_Ta réponse ici :_

> 

## Item 10 : Mesure de latence, coût en tokens, comparaison à un baseline

**Consignes :**
1. Écris une fonction `repondre_contexte_complet(user_id, question)` qui récupère **tout** l'historique brut (`memory.get_all`) et le passe intégralement au LLM, sans passer par `memory.search`.
2. Sur un même jeu de 5 questions, mesure pour chaque approche (`repondre_avec_memoire` vs `repondre_contexte_complet`) :
   - la latence (`time.perf_counter()`),
   - le nombre de tokens envoyés au LLM (`tiktoken`).
3. Affiche un graphique comparatif (barres) des deux approches.
4. Mets ces résultats en regard des chiffres du papier de recherche (91 % de réduction de latence p95, plus de 90 % de réduction de tokens) : est-on dans le même ordre de grandeur ? Si non, pourquoi (taille de la base de mémoire, longueur des conversations testées...) ?


In [7]:
import time
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")

def repondre_contexte_complet(user_id: str, question: str) -> str:
    tout = memory.get_all(user_id=user_id)
    tout = tout.get("results", tout)
    contexte = ", ".join([r.get("memory") for r in tout])
    instructions = system_prompt + contexte
    response = client.responses.create(model="gpt-4o-mini", instructions=instructions, input=question)
    return response.output_text

def mesurer(fonction, *args, **kwargs):
    debut = time.perf_counter()
    resultat = fonction(*args, **kwargs)
    return resultat, time.perf_counter() - debut

def compter_tokens_memoire(question, user_id="holding_corp"):
    results = memory.search(question, filters={"user_id": user_id})
    results = results.get("results", results)
    contexte = ", ".join([r.get("memory") for r in results[:4]])
    return len(enc.encode(system_prompt + contexte + question))

def compter_tokens_contexte_complet(question, user_id="holding_corp"):
    tout = memory.get_all(user_id=user_id)
    tout = tout.get("results", tout)
    contexte = ", ".join([r.get("memory") for r in tout])
    return len(enc.encode(system_prompt + contexte + question))


NameError: name 'jeu_de_donnees' is not defined

In [ ]:
# TODO — boucle de mesure sur ton jeu de 5 questions, stocke durées et tokens pour chaque approche

questions_test = [d["question"] for d in jeu_de_donnees[:5]]
resultats = {"memoire": [], "contexte_complet": []}

for q in questions_test:
    _, t_mem = mesurer(generer_reponse, q)
    resultats["memoire"].append({"latence": t_mem, "tokens": compter_tokens_memoire(q)})

    _, t_ctx = mesurer(repondre_contexte_complet, "holding_corp", q)
    resultats["contexte_complet"].append({"latence": t_ctx, "tokens": compter_tokens_contexte_complet(q)})

In [ ]:
import matplotlib.pyplot as plt

# TODO — trace un graphique en barres comparant les deux approches (latence ET tokens, 2 sous-graphiques par exemple)

lat_mem = [r["latence"] for r in resultats["memoire"]]
lat_ctx = [r["latence"] for r in resultats["contexte_complet"]]
tok_mem = [r["tokens"] for r in resultats["memoire"]]
tok_ctx = [r["tokens"] for r in resultats["contexte_complet"]]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].bar(["memory.search", "contexte complet"], [sum(lat_mem)/len(lat_mem), sum(lat_ctx)/len(lat_ctx)])
axes[0].set_title("Latence moyenne (s)")
axes[1].bar(["memory.search", "contexte complet"], [sum(tok_mem)/len(tok_mem), sum(tok_ctx)/len(tok_ctx)])
axes[1].set_title("Tokens moyens envoyés")
plt.tight_layout()
plt.show()

**🔍 Constat** — *Qu'observes-tu exactement ? Décris factuellement ce que tu vois (sans interpréter encore) dans la cellule ci-dessous.*

_Ta réponse ici :_

> 

**💬 Interprétation** — *Pourquoi ce comportement se produit-il, à ton avis ? Relie-le au fonctionnement interne de Mem0 vu dans le résumé de l'article.*

_Compare tes chiffres à ceux du papier de recherche. Explique au moins un facteur qui pourrait expliquer un écart entre tes résultats et les leurs._

_Ta réponse ici :_

> 

## Item 11 : Bilan

**Résumé, en quelques paragraphes :**

1. Ce qu'apporte concrètement une architecture comme Mem0 par rapport à un simple historique de conversation passé en entier au LLM.
2. Les exercices où un comportement inattendu a été observé, et ce que cet écart entre attente et résultat a appris.
3. Pour le projet d'agent WhatsApp business : 2 ou 3 endroits précis où réutiliser un des mécanismes vus ici (mémoire multi-utilisateurs, outil externe, mesure de latence...), et pourquoi.


_Ta réponse ici :_

> 